In [2]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path("/scratch/jczhang/submeso_data_driven").resolve()))
import numpy as np
import xarray as xr
import h5netcdf

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

from src.utils import read_mitgcm
from src.utils import make_grid
from src.utils import build_coarse_grid
from src.utils.coarsen_UVWTS import coarse_field

# --------------------- your submesoscale heat flux (kept same structure) -------------------
def submeso_vertical_heat_flux(
    ds_fine: xr.Dataset,
    ds_coarse: xr.Dataset,
    ds_ocean: xr.Dataset,
    ds_ocean_c: xr.Dataset,
    WT: xr.DataArray,
    boundary_Z: str = "fill",
    box_j: int = 10,
    box_i: int = 10,
    to_wm2=True,
):
    ds = xr.merge([ds_fine, ds_ocean])
    grid_fine = make_grid.make_grid_periodic_x(ds)
    grid_coarse = make_grid.make_grid_periodic_x(ds_coarse)

    upper = ds["hFacC"]
    lower = ds["hFacC"].shift(k=-1, fill_value=0.0)
    open_frac = xr.ufuncs.minimum(upper, lower).rename({"k": "k_l"})
    Aface = (open_frac * ds["RAC"]).astype("float32")
    wet_w = Aface > 0

    # kept here as in your original function
    T_w = grid_fine.interp(ds["T"], "Z", to="left", boundary=boundary_Z)

    def _area_weighted_boxmean(A: xr.DataArray, WGT: xr.DataArray, box_j=10, box_i=10) -> xr.DataArray:
        num = (A * WGT).coarsen(j=box_j, i=box_i, boundary="trim").sum()
        den = (WGT).coarsen(j=box_j, i=box_i, boundary="trim").sum()
        return xr.where(den > 0, num / den, np.nan)

    def as_ordinal(da: xr.DataArray) -> xr.DataArray:
        da = da.reset_coords(drop=True)
        new_coords = {d: xr.DataArray(np.arange(da.sizes[d]), dims=d) for d in da.dims}
        return da.assign_coords(**new_coords)

    # total coarse flux from supplied WT
    wT_bar_kL = _area_weighted_boxmean(WT.where(wet_w), Aface, box_j=box_j, box_i=box_i)
    w_bar_kL = ds_ocean_c["W"]
    T_bar_kC = ds_ocean_c["T"]
    T_bar_kL = grid_coarse.interp(T_bar_kC, "Z", to="left", boundary=boundary_Z)
    wT_bar_kL = as_ordinal(wT_bar_kL)

    # resolved/coarse part
    wT_meso_kL = (w_bar_kL * T_bar_kL)
    wT_sub_kL = (wT_bar_kL - wT_meso_kL).rename("wT_sub_kL")

    wT_bar_kL = wT_bar_kL.rename("wT_bar_kL")
    wT_meso_kL = wT_meso_kL.rename("wT_meso_kL")
    w_bar_kL = w_bar_kL.rename("w_bar_kL")
    T_bar_kL = T_bar_kL.rename("T_bar_kL")

    # interface -> center
    wT_sub_kC = grid_coarse.interp(wT_sub_kL, "Z", to="center", boundary=boundary_Z).rename("wT_sub_kC")
    wT_meso_kC = grid_coarse.interp(wT_meso_kL, "Z", to="center", boundary=boundary_Z).rename("wT_meso_kC")
    wT_bar_kC = grid_coarse.interp(wT_bar_kL, "Z", to="center", boundary=boundary_Z).rename("wT_bar_kC")

    if to_wm2:
        rho0, cp = 1035.0, 3994.0
        wT_meso_kC = (rho0 * cp * wT_meso_kC).assign_attrs(units="W m^-2")
        wT_bar_kC = (rho0 * cp * wT_bar_kC).assign_attrs(units="W m^-2")
        wT_sub_kC = (rho0 * cp * wT_sub_kC).assign_attrs(units="W m^-2")
    else:
        wT_meso_kC.attrs["units"] = "K m s^-1"
        wT_bar_kC.attrs["units"] = "K m s^-1"
        wT_sub_kC.attrs["units"] = "K m s^-1"

    wT_meso_kC.attrs["long_name"] = "mesoscale vertical heat flux at centers (interp from interfaces)"
    wT_bar_kC.attrs["long_name"] = "total coarse vertical heat flux at centers (interp from interfaces)"
    wT_sub_kC.attrs["long_name"] = "submesoscale vertical heat flux at centers (interp from interfaces)"

    return wT_sub_kC


# --------------------- loader ----------------------------------
def make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var: str, dims):
    def load_snapshot(it: int) -> xr.DataArray:
        da = read_mitgcm._ocean_var(grid_dir, var, it, dims)
        da = da.drop_vars("time", errors="ignore")
        da.name = var
        return da
    return load_snapshot


# --------------------- main worker ----------------------------------
def update_12h_submeso_flux_to_netcdf(
    start_time_index: int,
    end_time_index: int,
    load_W,
    load_T,
    out_nc: str,
    ds_fine: xr.Dataset,
    ds_coarse: xr.Dataset,
    box_j: int = 10,
    box_i: int = 10,
    block_size: int = 12,
    step: int = 144,
    time_name: str = "time",
    boundary_Z: str = "fill",
    to_wm2: bool = True,
    dtype=np.float32,
    skip_missing: bool = False,
    print_every: int = 20,
):
    start_time_index = int(start_time_index)
    end_time_index = int(end_time_index)
    step = int(step)

    if end_time_index < start_time_index:
        raise ValueError("end_time_index must be >= start_time_index")

    os.makedirs(os.path.dirname(out_nc) or ".", exist_ok=True)

    block_span = block_size * step

    saved = set()
    origin_iter = start_time_index

    if os.path.exists(out_nc):
        with h5netcdf.File(out_nc, "r") as nc:
            if time_name in nc.variables:
                saved = set(map(int, nc.variables[time_name][:].tolist()))
            if "origin_iter" in nc.attrs:
                origin_iter = int(nc.attrs["origin_iter"])
            if "box_j" in nc.attrs and int(nc.attrs["box_j"]) != int(box_j):
                raise ValueError("box_j mismatch with existing file")
            if "box_i" in nc.attrs and int(nc.attrs["box_i"]) != int(box_i):
                raise ValueError("box_i mismatch with existing file")

    # n-range for complete blocks
    n0 = int(np.ceil((start_time_index - origin_iter) / block_span)) if start_time_index > origin_iter else 0
    n1 = int(np.floor((end_time_index - origin_iter - (block_size - 1) * step) / block_span))
    if n1 < n0:
        print("Not enough hourly data to form even one full 12-hour block yet.")
        return

    # build fine grid once
    grid_fine = None

    def _init_file(example_flux_kC: xr.DataArray):
        spatial_dims = tuple(example_flux_kC.dims)
        spatial_sizes = {d: int(example_flux_kC.sizes[d]) for d in spatial_dims}

        with h5netcdf.File(out_nc, "w") as nc:
            nc.dimensions[time_name] = None
            for d, n in spatial_sizes.items():
                nc.dimensions[d] = n

            nc.attrs["origin_iter"] = int(origin_iter)
            nc.attrs["hourly_step"] = int(step)
            nc.attrs["block_size"] = int(block_size)
            nc.attrs["time_is_block_start"] = 1
            nc.attrs["box_j"] = int(box_j)
            nc.attrs["box_i"] = int(box_i)
            nc.attrs["variable"] = "wT_sub_kC"
            nc.attrs["to_wm2"] = int(bool(to_wm2))
            nc.attrs["total_flux_definition"] = "mean(W,12h) * interp_Zleft(mean(T,12h))"

            tvar = nc.create_variable(
                time_name,
                (time_name,),
                dtype="i8",
                chunks=(1,),
                maxshape=(None,),
            )
            tvar.attrs["long_name"] = "start index of each 12-hour averaging block"
            tvar.attrs["units"] = "model_iteration_index"

            dims = (time_name,) + spatial_dims
            chunks = (1,) + tuple(spatial_sizes[d] for d in spatial_dims)
            maxshape = (None,) + tuple(spatial_sizes[d] for d in spatial_dims)

            v = nc.create_variable(
                "wT_sub_kC",
                dims,
                dtype="f4",
                chunks=chunks,
                maxshape=maxshape,
            )
            v.attrs["long_name"] = "submesoscale vertical heat flux at centers (12h block)"
            v.attrs["units"] = "W m^-2" if to_wm2 else "K m s^-1"

    appended = skipped_saved = skipped_missing = 0
    nc_app = None

    try:
        for n in range(n0, n1 + 1):
            s = origin_iter + n * block_span

            if print_every and ((n - n0) % print_every == 0):
                print(f"[try] n={n}/{n1} block_start={s}", flush=True)

            if s in saved:
                skipped_saved += 1
                continue

            block_iters = [s + m * step for m in range(block_size)]

            W0 = T0 = None
            sumW = sumT = None

            try:
                for it in block_iters:
                    W = load_W(int(it)).astype(dtype)
                    T = load_T(int(it)).astype(dtype)

                    if W0 is None:
                        W0, T0 = W, T
                        sumW = W.values.astype(dtype, copy=True)
                        sumT = T.values.astype(dtype, copy=True)

                        if grid_fine is None:
                            ds_tmp = xr.merge([ds_fine, xr.Dataset({"W": W0, "T": T0})])
                            grid_fine = make_grid.make_grid_periodic_x(ds_tmp)
                    else:
                        sumW += W.values
                        sumT += T.values

            except Exception as e:
                if skip_missing:
                    skipped_missing += 1
                    print(f"  -> missing/unreadable snapshot in block_start={s}, skip ({e})", flush=True)
                    continue
                raise

            # ---------------- 12h means ----------------
            W_12mean = W0.copy(data=(sumW / block_size)).astype("float32")
            T_12mean = T0.copy(data=(sumT / block_size)).astype("float32")

            # ---------------- changed part ----------------
            # total flux = (12h mean W) * interp_Zleft(12h mean T)
            T_12mean_w = grid_fine.interp(T_12mean, "Z", to="left", boundary=boundary_Z)
            WT_12mean = (W_12mean * T_12mean_w).astype("float32")
            WT_12mean.name = "WT"

            # build ds_ocean and ds_ocean_c
            ds_ocean = xr.Dataset(dict(W=W_12mean, T=T_12mean))
            W_coarse = coarse_field("W", W_12mean, ds_fine, box_j=box_j, box_i=box_i)
            T_coarse = coarse_field("T", T_12mean, ds_fine, box_j=box_j, box_i=box_i)
            ds_ocean_c = xr.Dataset(dict(W=W_coarse, T=T_coarse))

            # compute submesoscale flux
            wT_sub_kC = submeso_vertical_heat_flux(
                ds_fine=ds_fine,
                ds_coarse=ds_coarse,
                ds_ocean=ds_ocean,
                ds_ocean_c=ds_ocean_c,
                WT=WT_12mean,
                boundary_Z=boundary_Z,
                box_j=box_j,
                box_i=box_i,
                to_wm2=to_wm2,
            ).astype("float32")

            # init file on first successful output
            if not os.path.exists(out_nc):
                _init_file(wT_sub_kC)

            if nc_app is None:
                nc_app = h5netcdf.File(out_nc, "a")

            tvar = nc_app.variables[time_name]
            write_idx = int(tvar.shape[0])

            v = nc_app.variables["wT_sub_kC"]
            spatial_shape = v.shape[1:]

            v._h5ds.resize((write_idx + 1,) + spatial_shape)
            v[write_idx, ...] = wT_sub_kC.values.astype(np.float32, copy=False)

            tvar._h5ds.resize((write_idx + 1,))
            tvar[write_idx] = s

            saved.add(s)
            appended += 1

    finally:
        if nc_app is not None:
            nc_app.close()

    print(
        f"\nUpdated NetCDF (saved only submeso flux): {out_nc}\n"
        f"  appended: {appended}\n"
        f"  skipped (already saved): {skipped_saved}\n"
        f"  skipped (missing hours): {skipped_missing}\n"
        f"  origin_iter={origin_iter}, step={step}, block_size={block_size}\n"
        f"  coarse box: box_j={box_j}, box_i={box_i}",
        flush=True,
    )


# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    path_scratch = "/scratch/jczhang/"
    path_MITgcm = "MITgcm_baroclinic_instability_old/MITgcm/verification/tutorial_reentrant_channel/run"
    grid_dir = path_scratch + path_MITgcm

    ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)

    box_j, box_i = 10, 10
    ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine, box_j=box_j, box_i=box_i)

    dims_map = {
        "W": ("k_l", "j", "i"),
        "T": ("k", "j", "i"),
    }

    start_time_index, end_time_index, step = 26622432, 26622432 + 12 * 144, 144

    load_W = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var="W", dims=dims_map["W"])
    load_T = make_mitgcm_loader_onevar(grid_dir, read_mitgcm, var="T", dims=dims_map["T"])

    out_nc = "./with_wind.nc"

    update_12h_submeso_flux_to_netcdf(
        start_time_index=start_time_index,
        end_time_index=end_time_index,
        load_W=load_W,
        load_T=load_T,
        out_nc=out_nc,
        ds_fine=ds_fine,
        ds_coarse=ds_coarse,
        box_j=box_j,
        box_i=box_i,
        block_size=12,
        step=step,
        time_name="time",
        boundary_Z="fill",
        to_wm2=True,
        skip_missing=False,
        print_every=1,
    )

[try] n=0/0 block_start=26622432

Updated NetCDF (saved only submeso flux): ./with_wind.nc
  appended: 1
  skipped (already saved): 0
  skipped (missing hours): 0
  origin_iter=26622432, step=144, block_size=12
  coarse box: box_j=10, box_i=10
